<a href="https://colab.research.google.com/github/Ricklessfun/RICK/blob/main/%E2%80%9C%E2%80%9CEvolvePro%E2%80%94%E2%80%94multi_mutants_Round2_ipynb%E2%80%9D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup:

In [ ]:
!git clone https://github.com/mat10d/EvolvePro.git
%cd EvolvePro/

Cloning into 'EvolvePro'...
remote: Enumerating objects: 1387, done.
remote: Counting objects: 100% (241/241), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 1387 (delta 198), reused 172 (delta 172), pack-reused 1146 (from 2)
Receiving objects: 100% (1387/1387), 85.14 MiB | 19.20 MiB/s, done.
Resolving deltas: 100% (720/720), done.
/content/EvolvePro


In [ ]:
%%capture

!pip install pandas numpy scikit-learn scikit-learn-extra xgboost matplotlib seaborn biopython scipy torch fair-esm
!mkdir /content/output

## Process

In [ ]:
from evolvepro.src.process import generate_wt, generate_single_aa_mutants
generate_wt('MGKSKEISQDLRKRIVDLHKSGSSLGAISKRLAVPRSSVQTIVRKYKHHGTTQPSYRSGRRRVLSPRDERTLVRKVQINPRTTAKDLVKMLEETGTKVSISTVKRVLYRHNLKGHSARKKPLLQNRHKKARLRFATAHGDKDRTFWRNVLWSDETKIELFGHNDHRYVWRKKGEASKPKNTIPTVKHGGGSIMLWGCFAAGGTGALHKIDGSMDAVQYVDILKQHLKTSVRKLKLGRKWVFQHDNDPKHTSKVVAKWLKDNKVKVLEWPSQSPDLNPIENLWAELKKRVRARRPTNLTQLHQLCQEEWAKIHPNYCGKLVEGYPKRLTQVKQFKGNATKY', output_file='/content/output/hsSB_WT.fasta')
generate_single_aa_mutants('/content/output/hsSB_WT.fasta', output_file='/content/output/hsSB.fasta')

/content/EvolvePro/evolvepro/src/process.py:471: SyntaxWarning: invalid escape sequence '\d'
  "(\d+)([A-Z]+)", expand=True


Number of mutants: 6461


In [ ]:
from evolvepro.src.process import suggest_initial_mutants
suggest_initial_mutants('/content/output/hsSB.fasta', num_mutants=16, random_seed=42)


Suggested 16 mutants for testing:
1. K141T
2. D260W
3. Y340H
4. T155I
5. G139A
6. I7W
7. R31S
8. V265Y
9. R44M
10. P294E
11. K264W
12. H127W
13. Y56A
14. S55P
15. R290Y
16. R62C


## PLM

In [ ]:
!python evolvepro/plm/esm/extract.py esm1b_t33_650M_UR50S /content/output/hsSB.fasta /content/output/hsSB_esm1b_t33_650M_UR50S --toks_per_batch 512 --include mean --concatenate_dir /content/output

In [ ]:
!unzip "/content/output/hsSB_esm2_t48_15B_UR50D.zip" -d /content/output/

In [ ]:
from evolvepro.src.evolve import evolve_experimental

protein_name = 'hsSB'
embeddings_base_path = '/content/output'
embeddings_file_name = 'hsSB_esm2_t48_15B_UR50D.csv'
round_base_path = '/content/EvolvePro/colab/rounds_data'
wt_fasta_path = "/content/output/hsSB_WT.fasta"
number_of_variants = 16
output_dir = '/content/output/'
rename_WT = True

#### Round 1

In [ ]:
round_name = 'Round1'
round_file_names = ['hsSB_Round1.xlsx']

this_round_variants, df_test, df_sorted_all = evolve_experimental(
    protein_name,
    round_name,
    embeddings_base_path,
    embeddings_file_name,
    round_base_path,
    round_file_names,
    wt_fasta_path,
    rename_WT,
    number_of_variants,
    output_dir
)

Processing hsSB - Round1
Embeddings loaded: (6461, 5120)
Loaded experimental data for hsSB_Round1.xlsx: (16, 3)
iteration shape: (16, 2)
Labels shape: (6461, 5)
Embeddings and labels are aligned
(6445,)

Tested variants in this round: 16
606     D260W
1045    G139A
1499    H127W
1993      I7W
2143    K141T
2506    K264W
3879    P294E
4578    R290Y
4631     R31S
4684     R44M
4751     R62C
5180     S55P
5327    T155I
5927    V265Y
6410    Y340H
6442     Y56A
Name: variant, dtype: object

Top 16 variants predicted by the model:
     variant    y_pred  y_actual  y_actual_scaled  y_actual_binary  \
2651   K334N  1.037041       NaN              NaN              NaN   
5235     S8M  1.035849       NaN              NaN              NaN   
5280   T102W  1.030643       NaN              NaN              NaN   
5229     S8F  1.028431       NaN              NaN              NaN   
4270     Q9S  1.023446       NaN              NaN              NaN   
2933   L132I  1.023035       NaN              Na

/content/EvolvePro/evolvepro/src/model.py:303: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat([df_train, df_test])


#### Round 2

In [ ]:
round_name = 'Round2'
round_file_names = ['hsSB_Round1.xlsx', 'hsSB_Round2.xlsx']

this_round_variants, df_test, df_sorted_all = evolve_experimental(
    protein_name,
    round_name,
    embeddings_base_path,
    embeddings_file_name,
    round_base_path,
    round_file_names,
    wt_fasta_path,
    rename_WT,
    number_of_variants,
    output_dir
)

Processing hsSB - Round2
Embeddings loaded: (6461, 5120)
Loaded experimental data for hsSB_Round1.xlsx: (16, 3)
Loaded experimental data for hsSB_Round2.xlsx: (16, 3)
iteration shape: (32, 2)
Labels shape: (6461, 5)
Embeddings and labels are aligned
(6429,)

Tested variants in this round: 32
486      D17P
489      D17S
606     D260W
1045    G139A
1499    H127W
1993      I7W
2143    K141T
2375    K238A
2390    K238T
2471    K262C
2490    K264C
2506    K264W
2651    K334N
2897    L122M
2933    L132I
3535    N148C
3879    P294E
4270      Q9S
4578    R290Y
4631     R31S
4684     R44M
4751     R62C
5036    S251C
5180     S55P
5229      S8F
5235      S8M
5280    T102W
5327    T155I
5434    T295A
5927    V265Y
6410    Y340H
6442     Y56A
Name: variant, dtype: object

Top 16 variants predicted by the model:
     variant    y_pred  y_actual  y_actual_scaled  y_actual_binary  \
4267     Q9N  1.131584       NaN              NaN              NaN   
2225   K179D  1.126162       NaN              NaN

/content/EvolvePro/evolvepro/src/model.py:303: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat([df_train, df_test])


#### Round 3

In [ ]:
round_name = 'Round3'
round_file_names = ['hsSB_Round1.xlsx', 'hsSB_Round2.xlsx', 'hsSB_Round3.xlsx']

this_round_variants, df_test, df_sorted_all = evolve_experimental(
    protein_name,
    round_name,
    embeddings_base_path,
    embeddings_file_name,
    round_base_path,
    round_file_names,
    wt_fasta_path,
    rename_WT,
    number_of_variants,
    output_dir
)

#### Round 4

In [ ]:
round_name = 'Round4'
round_file_names = ['hsSB_Round1.xlsx', 'hsSB_Round2.xlsx', 'hsSB_Round3.xlsx', 'hsSB_Round4.xlsx']

this_round_variants, df_test, df_sorted_all = evolve_experimental(
    protein_name,
    round_name,
    embeddings_base_path,
    embeddings_file_name,
    round_base_path,
    round_file_names,
    wt_fasta_path,
    rename_WT,
    number_of_variants,
    output_dir
)

#### Round 5

In [ ]:
round_name = 'Round5'
round_file_names = ['hsSB_Round1.xlsx', 'hsSB_Round2.xlsx', 'hsSB_Round3.xlsx', 'hsSB_Round4.xlsx', 'hsSB_Round5.xlsx']

this_round_variants, df_test, df_sorted_all = evolve_experimental(
    protein_name,
    round_name,
    embeddings_base_path,
    embeddings_file_name,
    round_base_path,
    round_file_names,
    wt_fasta_path,
    rename_WT,
    number_of_variants,
    output_dir
)

In [ ]:
# ==============================
# 模块1：准备多点突变数据和嵌入向量
# ==============================
print("="*60)
print("模块1：准备多点突变数据和嵌入向量")
print("="*60)

import os
import pandas as pd
from evolvepro.src.process import generate_n_mutant_combinations

# 配置参数
output_dir = '/content/output'
round_base_path = '/content/EvolvePro/colab/rounds_data'
wt_fasta_path = '/content/output/hsSB_WT.fasta'
protein_name = 'hsSB'
ACTIVITY_THRESHOLD = 1.19

# 1. 准备有益突变数据
print("1. 准备有益突变数据...")
all_rounds_data = []
for i in range(1, 6):
    round_file = os.path.join(round_base_path, f'{protein_name}_Round{i}.xlsx')
    if os.path.exists(round_file):
        round_data = pd.read_excel(round_file)
        all_rounds_data.append(round_data)
        print(f"Loaded {protein_name}_Round{i}.xlsx: {len(round_data)} variants")

if not all_rounds_data:
    raise FileNotFoundError("未找到前5轮实验数据文件")

combined_data = pd.concat(all_rounds_data, ignore_index=True)
beneficial_mutations = combined_data[combined_data['activity'] >= ACTIVITY_THRESHOLD].copy()
print(f"有益突变数量: {len(beneficial_mutations)}")

beneficial_file = os.path.join(output_dir, 'beneficial_mutations.xlsx')
beneficial_mutations[['Variant', 'activity']].to_excel(beneficial_file, index=False)

# 2. 生成多点突变FASTA
print("2. 生成多点突变FASTA...")
double_fasta = os.path.join(output_dir, f'{protein_name}_double_mutants.fasta')
triple_fasta = os.path.join(output_dir, f'{protein_name}_triple_mutants.fasta')

generate_n_mutant_combinations(
    wt_fasta=wt_fasta_path,
    mutant_file=beneficial_file,
    n=2,
    output_file=double_fasta,
    threshold=ACTIVITY_THRESHOLD
)

generate_n_mutant_combinations(
    wt_fasta=wt_fasta_path,
    mutant_file=beneficial_file,
    n=3,
    output_file=triple_fasta,
    threshold=ACTIVITY_THRESHOLD
)

# 3. 提取PLM嵌入向量
print("3. 提取PLM嵌入向量...")
!python evolvepro/plm/esm/extract.py esm1b_t33_650M_UR50S {double_fasta} /content/output/hsSB_double_esm1b_t33_650M_UR50S --toks_per_batch 512 --include mean --concatenate_dir /content/output
!python evolvepro/plm/esm/extract.py esm1b_t33_650M_UR50S {triple_fasta} /content/output/hsSB_triple_esm1b_t33_650M_UR50S --toks_per_batch 512 --include mean --concatenate_dir /content/output

print("✓ 模块1完成：多点突变数据和嵌入向量准备就绪")

In [ ]:
 # ==============================
# 模块2：第6轮多点突变进化
# ==============================
print("="*60)
print("模块2：第6轮多点突变进化")
print("="*60)

from evolvepro.src.evolve import evolve_experimental_multi

# 配置参数
protein_name = 'hsSB'
embeddings_base_path = '/content/output'
round_base_path = '/content/EvolvePro/colab/rounds_data'
wt_fasta_path = '/content/output/hsSB_WT.fasta'
number_of_variants = 12
output_dir = '/content/output/'
rename_WT = True

# 第6轮参数
round_name = 'Round6'
embeddings_file_names = [
    'hsSB_esm1b_t33_650M_UR50S.csv',
    'hsSB_double_mutants_esm1b_t33_650M_UR50S.csv',
    'hsSB_triple_mutants_esm1b_t33_650M_UR50S.csv'
]
round_file_names_single = ['hsSB_Round1.xlsx', 'hsSB_Round2.xlsx', 'hsSB_Round3.xlsx', 'hsSB_Round4.xlsx', 'hsSB_Round5.xlsx']
round_file_names_multi = []  # 第一轮多点突变，没有之前的多点数据

print(f"Processing {protein_name} - {round_name}")
print(f"Embeddings: {embeddings_file_names}")

# 执行第6轮进化
this_round_variants, df_test, df_sorted_all = evolve_experimental_multi(
    protein_name,
    round_name,
    embeddings_base_path,
    embeddings_file_names,
    round_base_path,
    round_file_names_single,
    round_file_names_multi,
    wt_fasta_path,
    rename_WT,
    number_of_variants,
    output_dir
)

print("✓ 第6轮完成！推荐结果已生成")
print("请进行湿实验并将结果保存为: hsSB_Round6.xlsx")

In [ ]:
# ==============================
# 模块3：第7轮多点突变进化
# ==============================
print("="*60)
print("模块3：第7轮多点突变进化")
print("="*60)

from evolvepro.src.evolve import evolve_experimental_multi

# 配置参数（同上）
protein_name = 'hsSB'
embeddings_base_path = '/content/output'
round_base_path = '/content/EvolvePro/colab/rounds_data'
wt_fasta_path = '/content/output/hsSB_WT.fasta'
number_of_variants = 12
output_dir = '/content/output/'
rename_WT = True

# 第7轮参数
round_name = 'Round7'
embeddings_file_names = [
    'hsSB_esm1b_t33_650M_UR50S.csv',
    'hsSB_double_mutants_esm1b_t33_650M_UR50S.csv',
    'hsSB_triple_mutants_esm1b_t33_650M_UR50S.csv'
]
round_file_names_single = ['hsSB_Round1.xlsx', 'hsSB_Round2.xlsx', 'hsSB_Round3.xlsx', 'hsSB_Round4.xlsx', 'hsSB_Round5.xlsx']
round_file_names_multi = ['hsSB_Round6.xlsx']  # 加入第6轮数据

print(f"Processing {protein_name} - {round_name}")
print(f"Embeddings: {embeddings_file_names}")

# 执行第7轮进化
this_round_variants, df_test, df_sorted_all = evolve_experimental_multi(
    protein_name,
    round_name,
    embeddings_base_path,
    embeddings_file_names,
    round_base_path,
    round_file_names_single,
    round_file_names_multi,
    wt_fasta_path,
    rename_WT,
    number_of_variants,
    output_dir
)

print("✓ 第7轮完成！推荐结果已生成")
print("请进行湿实验并将结果保存为: hsSB_Round7.xlsx")

In [ ]:
# ==============================
# 模块4：第8轮多点突变进化
# ==============================
print("="*60)
print("模块4：第8轮多点突变进化")
print("="*60)

from evolvepro.src.evolve import evolve_experimental_multi

# 配置参数（同上）
protein_name = 'hsSB'
embeddings_base_path = '/content/output'
round_base_path = '/content/EvolvePro/colab/rounds_data'
wt_fasta_path = '/content/output/hsSB_WT.fasta'
number_of_variants = 12
output_dir = '/content/output/'
rename_WT = True

# 第8轮参数
round_name = 'Round8'
embeddings_file_names = [
    'hsSB_esm1b_t33_650M_UR50S.csv',
    'hsSB_double_mutants_esm1b_t33_650M_UR50S.csv',
    'hsSB_triple_mutants_esm1b_t33_650M_UR50S.csv'
]
round_file_names_single = ['hsSB_Round1.xlsx', 'hsSB_Round2.xlsx', 'hsSB_Round3.xlsx', 'hsSB_Round4.xlsx', 'hsSB_Round5.xlsx']
round_file_names_multi = ['hsSB_Round6.xlsx', 'hsSB_Round7.xlsx']  # 加入第6-7轮数据

print(f"Processing {protein_name} - {round_name}")
print(f"Embeddings: {embeddings_file_names}")

# 执行第8轮进化
this_round_variants, df_test, df_sorted_all = evolve_experimental_multi(
    protein_name,
    round_name,
    embeddings_base_path,
    embeddings_file_names,
    round_base_path,
    round_file_names_single,
    round_file_names_multi,
    wt_fasta_path,
    rename_WT,
    number_of_variants,
    output_dir
)

print("✓ 第8轮完成！推荐结果已生成")
print("请进行湿实验并将结果保存为: hsSB_Round8.xlsx")

## Plot

In [ ]:
"""## 热点位点探索热图（简洁版）"""

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 简洁的热图生成函数
def plot_simple_heatmap(round_files, round_base_path, output_dir, protein_name):
    """生成简洁的热点位点探索热图"""

    # 收集所有突变位点数据
    position_data = {}

    for i, round_file in enumerate(round_files, 1):
        file_path = os.path.join(round_base_path, round_file)
        if not os.path.exists(file_path):
            continue

        df = pd.read_excel(file_path)
        round_name = f'R{i}'
        position_data[round_name] = {}

        for variant in df['Variant']:
            # 提取位点数字
            digits = ''.join(filter(str.isdigit, str(variant)))
            if digits:
                pos = int(digits)
                position_data[round_name][pos] = position_data[round_name].get(pos, 0) + 1

    # 转换为DataFrame
    all_positions = sorted(set(pos for rd in position_data.values() for pos in rd.keys()))
    heatmap_data = []

    for pos in all_positions:
        row = [position_data[round].get(pos, 0) for round in position_data.keys()]
        heatmap_data.append(row)

    heatmap_df = pd.DataFrame(heatmap_data, index=all_positions, columns=position_data.keys())

    # 只保留有探索的位点
    heatmap_df = heatmap_df[(heatmap_df > 0).any(axis=1)]

    if heatmap_df.empty:
        print("❌ 无数据可绘制")
        return

    # 绘制热图
    plt.figure(figsize=(10, 8))
    sns.heatmap(heatmap_df, cmap='YlOrRd', annot=True, fmt='d', linewidths=0.5)
    plt.title(f'{protein_name} - 残基探索热图')
    plt.xlabel('进化轮次')
    plt.ylabel('残基位置')

    # 保存图片
    plt.savefig(f'{output_dir}/{protein_name}_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ 热图已生成: {len(heatmap_df)}个位点 × {len(heatmap_df.columns)}轮次")

# 执行热图生成
round_files = [f'hsSB_Round{i}.xlsx' for i in range(1, 9)]
existing_files = [f for f in round_files if os.path.exists(os.path.join(round_base_path, f))]

if existing_files:
    plot_simple_heatmap(existing_files, round_base_path, output_dir, protein_name)
else:
    print("❌ 未找到任何数据文件")

In [ ]:
"""## 进化轨迹可视化（简洁版）"""

def plot_simple_evolution(round_files, round_base_path, wt_fasta_path, output_dir):
    """生成简洁的进化轨迹图"""

    # 收集所有数据
    all_data = []
    for round_file in round_files:
        file_path = os.path.join(round_base_path, round_file)
        if not os.path.exists(file_path):
            continue

        df = pd.read_excel(file_path)
        round_num = int(round_file.split('Round')[1].split('.')[0])
        df['round'] = round_num
        all_data.append(df)

    if not all_data:
        print("❌ 无数据可绘制")
        return

    # 合并数据
    combined_df = pd.concat(all_data, ignore_index=True)

    # 绘制进化轨迹
    plt.figure(figsize=(12, 6))

    # 按轮次绘制箱线图
    plt.subplot(1, 2, 1)
    sns.boxplot(data=combined_df, x='round', y='activity')
    plt.title('各轮次活性分布')
    plt.xlabel('进化轮次')
    plt.ylabel('活性')

    # 绘制最佳活性变化
    plt.subplot(1, 2, 2)
    best_activity = combined_df.groupby('round')['activity'].max()
    plt.plot(best_activity.index, best_activity.values, 'o-', linewidth=2, markersize=8)
    plt.title('最佳活性进化轨迹')
    plt.xlabel('进化轮次')
    plt.ylabel('最高活性')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{output_dir}/evolution_trajectory.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ 进化轨迹图已生成: {len(combined_df)}个数据点")

# 执行进化轨迹绘制
plot_simple_evolution(existing_files, round_base_path, wt_fasta_path, output_dir)

In [ ]:
from evolvepro.src.plot import read_exp_data, plot_variants_by_iteration

round_base_path = '/content/EvolvePro/colab/rounds_data'
round_file_names = ['hsSB_Round1.xlsx', 'hsSB_Round2.xlsx', 'hsSB_Round3.xlsx', 'hsSB_Round4.xlsx', 'hsSB_Round5.xlsx']
wt_fasta_path = "/content/output/hsSB_WT.fasta"

In [ ]:
df = read_exp_data(round_base_path, round_file_names, wt_fasta_path)
plot_variants_by_iteration(df, activity_column='activity', output_dir=output_dir, output_file="hsSB")
